This notebook uses the rio-stac API to create a STAC Item from the StormwaterHeatmap data catalog.

This section has additional metadata.
Not yet in the stac catalog yet.

In [21]:
import json
import requests

file_name = 'https://storage.googleapis.com/swhm-stac-data/layer_metadata/swhm_data_dict.json'

try:
    response = requests.get(file_name)
    response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
    data_dict = response.json()

    print(data_dict)
except requests.exceptions.RequestException as e:
    print(f"Error fetching data from URL: {e}")
except json.JSONDecodeError as e:
    print(f"Error decoding JSON from response: {e}")


{'Age of Imperviousness': {'default_reduction': ['Category'], 'description': ['Impervious surface by estimated date of conversion'], 'discrete': ['TRUE'], 'docs_link': ['docs/API/image_layers/Age%20of%20Development'], 'labels': [['2000–2014'], ['1990–2000'], ['Before 1990']], 'layer': {'eeObject': {'type': ['Image'], 'bands': [{'id': ['age_of_impervious_surface'], 'data_type': {'type': ['PixelType'], 'precision': ['int'], 'min': [0], 'max': [255]}, 'dimensions': [[31523], [30154]], 'crs': ['EPSG:3857'], 'crs_transform': [[15], [0], [-13904050.3428], [0], [-15], [6328206.6642]]}], 'version': [1752089904332127], 'id': ['projects/ee-swhm/assets/production_layers/Age_of_Imperviousness'], 'properties': {'pro2': ['[1, 2, 3]'], 'prop1': ['val1'], 'system:footprint': {'type': ['LineString'], 'coordinates': [[[-121.6501], [46.5929]], [[-121.3847], [46.5929]], [[-121.0528], [46.5928]], [[-120.6544], [46.5928]], [[-120.6544], [49.3135]], [[-121.0528], [49.3135]], [[-121.7165], [49.3135]], [[-122.

In [22]:
def unwrap_lists(obj):
    """
    Recursively unwrap single-element lists in a nested structure.
    Multi-element lists are preserved.
    """
    if isinstance(obj, dict):
        # Process each key-value pair in the dictionary
        return {key: unwrap_lists(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        # If it's a single-element list, unwrap it and process the element
        if len(obj) == 1:
            return unwrap_lists(obj[0])
        # If it's a multi-element list, process each element
        else:
            return [unwrap_lists(item) for item in obj]
    else:
        # Base case: return the value as-is
        return obj

# Normalize the entire data_dict
normalized_data_dict = {}
for layer_name, layer_data in data_dict.items():
    normalized_data_dict[layer_name] = unwrap_lists(layer_data)

In [ ]:
normalized_data_dict['Soils']

{'default_reduction': 'Category',
 'description': 'Hydrologic Soil Group',
 'discrete': 'TRUE',
 'labels': ['Outwash', 'Till', 'Saturated', 'Water'],
 'layer': {'eeObject': {'type': 'Image',
   'bands': {'id': 'soils',
    'data_type': {'type': 'PixelType',
     'precision': 'int',
     'min': 0,
     'max': 255},
    'dimensions': [121344, 126464],
    'crs': 'EPSG:3857',
    'crs_transform': [4.7773, 0, -13928661.0422, 0, -4.7773, 6419487.3835]},
   'version': 1683224707131757,
   'id': 'projects/ee-swhm/assets/production_layers/Soils',
   'properties': {'system:footprint': {'type': 'LineString',
     'coordinates': [[-120.6888, 46.2176],
      [-120.2819, 46.2177],
      [-119.9157, 46.2177],
      [-119.9157, 49.8451],
      [-120.5667, 49.8451],
      [-121.4618, 49.8451],
      [-122.4382, 49.8451],
      [-123.4146, 49.8451],
      [-124.3096, 49.8451],
      [-125.1233, 49.8451],
      [-125.1233, 46.2177],
      [-124.7164, 46.2177],
      [-124.1469, 46.2172],
      [-123.495

In [ ]:
import shutil
import os

if os.path.exists('./input_data_layers'):
    shutil.rmtree('./input_data_layers')
    print("Deleted existing 'input_data_layers' directory.")
else:
    print("'input_data_layers' directory does not exist.")

Deleted existing 'input_data_layers' directory.


This section saves the catalog to a local file

In [ ]:
import os
import pystac
from datetime import datetime
from rio_stac import create_stac_item

# --- 1. Configuration ---
# list of raw GS URIs
raw_uris = [
    "gs://live_data_layers/rasters/Age_of_Imperviousness.tif",
    #"gs://live_data_layers/rasters/Flow_Duration_Index.tif",
    "gs://live_data_layers/rasters/HSPF_Land_Cover_Type.tif",
    #"gs://live_data_layers/rasters/Hydrologic_Response_Units.tif",
    "gs://live_data_layers/rasters/Imperviousness.tif",
    "gs://live_data_layers/rasters/Land_Cover.tif",
    "gs://live_data_layers/rasters/Land_Use.tif",
    "gs://live_data_layers/rasters/Population_Density.tif",
    #"gs://live_data_layers/rasters/Precipitation_mm.tif",
    #"gs://live_data_layers/rasters/Runoff_mm.tif",
    "gs://live_data_layers/rasters/Slope.tif",
    "gs://live_data_layers/rasters/Slope_Categories.tif",
    "gs://live_data_layers/rasters/Soils.tif",
    #"gs://live_data_layers/rasters/Total_Copper_Concentration.tif",
    #"gs://live_data_layers/rasters/Total_Kjeldahl_Nitrogen_Concentration.tif",
    #"gs://live_data_layers/rasters/Total_Phosphorus_Concentration.tif",
    #"gs://live_data_layers/rasters/Total_Suspended_Solids_Concentration.tif",
    #"gs://live_data_layers/rasters/Total_Zinc_Concentration.tif",
    "gs://live_data_layers/rasters/Traffic.tif"
]

def get_thumbnail_url(uri):
    return uri.replace("gs://", "https://storage.googleapis.com/").replace(".tif", ".png").replace("/rasters/", "/thumbnails/")

# Convert gs:// to public https:// for web-readability
def gs_to_https(uri):
    return uri.replace("gs://", "https://storage.googleapis.com/")

def get_first(value):
    """Extract first element from list if it's a list, otherwise return value"""
    return value[0] if isinstance(value, list) and len(value) > 0 else value

output_dir = "./input_data_layers"
os.makedirs(output_dir, exist_ok=True)


In [ ]:
# --- Build lookup dictionary for efficiency ---
metadata_lookup = {}
for layer_name, layer_data in normalized_data_dict.items():
    # Get the layer title/name and replace spaces with underscores to match item_id
    title_name = layer_data.get('layer', {}).get('name', layer_name)
    if isinstance(title_name, str):
        title_name = title_name.replace(' ', '_')
        metadata_lookup[title_name] = layer_data

In [ ]:
print(metadata_lookup.keys())

dict_keys(['Age_of_Imperviousness', 'Flow_Duration_Index', 'HSPF_Land_Cover_Type', 'Hydrologic_Response_Units', 'Imperviousness', 'Land_Cover', 'Land_Use', 'Population_Density', 'Precipitation_(mm)', 'Runoff_(mm)', 'Slope', 'Slope_Categories', 'Soils', 'Traffic'])


In [ ]:
# --- Initialization (Ensure these exist) ---
catalog = pystac.Catalog(
    id="Stormwater Heatmap Data Catalog",
    description="""
                         The Puget Sound Stormwater Heatmap is an
                         open-source mapping tool and data repository to improve
                          understanding of stormwater pollution and to help guide
                          decisions to improve water quality. Data layers used in
                          the Stormwater Heatmap are stored as a SpatioTemporal
                          Asset Catalog (STAC) to improve accessibility and access.
                          """
)
collection = pystac.Collection(
    id="input_data_layers",
    description="Base data layers that are used to inform or develop other layers",
    extent=pystac.Extent(
        spatial=pystac.SpatialExtent([[-180, -90, 180, 90]]), # Placeholder: updated during normalize
        temporal=pystac.TemporalExtent([[datetime(2023, 1, 1), None]])
    )
)
catalog.add_child(collection)

<Link rel=child target=<Collection id=input_data_layers>>

In [32]:
# --- Process Items ---
print(f"Processing {len(raw_uris)} layers...")
print("-" * 60)

successful_items = []
failed_items = []

for uri in raw_uris:
    https_url = gs_to_https(uri)
    item_id = os.path.basename(uri).replace(".tif", "")
    thumbnail_url = get_thumbnail_url(uri)

    try:
        # Create STAC item with rio-stac (reads metadata via vsicurl)
        item = create_stac_item(
            source=https_url,
            input_datetime=datetime(2023, 1, 1),
            id=item_id,
            asset_name="data",
            asset_roles=["data"],
            asset_media_type=pystac.MediaType.COG,
            with_proj=True,
            with_raster=True
        )

        # Add thumbnail asset
        item.add_asset(
            key="thumbnail",
            asset=pystac.Asset(
                href=thumbnail_url,
                media_type=pystac.MediaType.PNG,
                roles=["thumbnail"],
                title="Thumbnail image"
            )
        )

        # Add metadata from normalized_data_dict
        metadata = metadata_lookup.get(item_id)
        if metadata:
            # Extract basic properties
            name = metadata.get('layer', {}).get('name', item_id)
            description = metadata.get('description', '')
            is_discrete = metadata.get('discrete', 'FALSE')

            item.properties['title'] = name
            item.properties['description'] = description

            # Add additional properties
            if 'sourceName' in metadata:
                item.properties['source'] = metadata['sourceName']
            if 'units' in metadata:
                item.properties['units'] = metadata['units']
            if 'scale' in metadata:
                item.properties['gsd'] = metadata['scale']  # Ground sample distance

            # Add classification for discrete data
            if is_discrete == 'TRUE':
                try:
                    values = metadata.get('values', [])
                    labels = metadata.get('labels', [])
                    palette = metadata.get('visParams', {}).get('palette', [])

                    classification_classes = []

                    # Build classification table
                    for i in range(len(values)):
                        value = values[i] if i < len(values) else str(i)
                        label = labels[i] if i < len(labels) else f"Class {value}"
                        color = palette[i] if i < len(palette) else "000000"

                        # Clean up color hex code
                        color = color.lstrip('#')

                        classification_classes.append({
                            'value': int(value),
                            'description': label,
                            'color_hint': f"#{color}"
                        })

                    if classification_classes:
                        # Add classification extension
                        item.stac_extensions.append(
                            "https://stac-extensions.github.io/classification/v1.1.0/schema.json"
                        )
                        item.assets['data'].extra_fields['classification:classes'] = classification_classes

                        print(f"   Added {len(classification_classes)} classification classes")

                except (KeyError, IndexError, ValueError, TypeError) as e:
                    print(f"⚠️  Could not add classification for {item_id}: {e}")
        else:
            print(f"⚠️  No metadata found for {item_id}")

        # Add item to collection
        collection.add_item(item)
        successful_items.append(item_id)
        print(f"✅ Created: {item_id}")

    except Exception as e:
        failed_items.append((item_id, str(e)))
        print(f"❌ Failed to process {item_id}: {e}")

# --- Finalize Catalog ---
print("-" * 60)
print(f"\nProcessing complete!")
print(f"  ✅ Successful: {len(successful_items)}")
print(f"  ❌ Failed: {len(failed_items)}")

if failed_items:
    print("\nFailed items:")
    for item_id, error in failed_items:
        print(f"  - {item_id}: {error}")

if successful_items:
    # Update extent from actual items
    collection.update_extent_from_items()

    # Normalize and save catalog
    print(f"\nSaving catalog to {output_dir}...")
    catalog.normalize_hrefs(output_dir)
    catalog.save(pystac.CatalogType.SELF_CONTAINED)

    print(f"✅ Catalog saved successfully!")
    print(f"   Location: {output_dir}")
    print(f"   Items: {len(list(collection.get_items()))}")
    print(f"   Catalog URL: {os.path.join(output_dir, 'catalog.json')}")
else:
    print("\n⚠️  No items were successfully processed. Catalog not saved.")

Processing 10 layers...
------------------------------------------------------------
   Added 3 classification classes
✅ Created: Age_of_Imperviousness
   Added 6 classification classes
✅ Created: HSPF_Land_Cover_Type
✅ Created: Imperviousness
   Added 8 classification classes
✅ Created: Land_Cover
   Added 15 classification classes
✅ Created: Land_Use
✅ Created: Population_Density
✅ Created: Slope
   Added 3 classification classes
✅ Created: Slope_Categories
   Added 4 classification classes
✅ Created: Soils
✅ Created: Traffic
------------------------------------------------------------

Processing complete!
  ✅ Successful: 10
  ❌ Failed: 0

Saving catalog to ./input_data_layers...
✅ Catalog saved successfully!
   Location: ./input_data_layers
   Items: 10
   Catalog URL: ./input_data_layers/catalog.json


In [ ]:
item_to_display = collection.get_item('soils')
item_to_display